In [1]:
import pandas as pd
from pathlib import Path

## Reading Data

In [5]:
gold_dir = Path("../../data/gold")
parquet_paths = list(gold_dir.rglob("*.parquet"))

dfs = {}
for p in parquet_paths:
    dfs[p.name] = pd.read_parquet(p)
    print(f"loaded {p.name} -> shape {dfs[p.name].shape}")


loaded embeddings_load_date=2026-03-12.parquet -> shape (11999, 4)
loaded embedding_features_load_date=2026-03-12.parquet -> shape (11999, 6)
loaded evidence_features_load_date=2026-03-12.parquet -> shape (11999, 13)
loaded lexical_features_load_date=2026-03-12.parquet -> shape (11999, 31)
loaded ml_features_load_date=2026-03-12.parquet -> shape (11999, 78)
loaded novelty_features_load_date=2026-03-12.parquet -> shape (11999, 7)
loaded product_context_load_date=2026-03-12.parquet -> shape (11999, 14)
loaded sentiment_features_load_date=2026-03-12.parquet -> shape (11999, 26)
loaded temporal_features_load_date=2026-03-12.parquet -> shape (11999, 31)


In [6]:
dfs.keys()

dict_keys(['embeddings_load_date=2026-03-12.parquet', 'embedding_features_load_date=2026-03-12.parquet', 'evidence_features_load_date=2026-03-12.parquet', 'lexical_features_load_date=2026-03-12.parquet', 'ml_features_load_date=2026-03-12.parquet', 'novelty_features_load_date=2026-03-12.parquet', 'product_context_load_date=2026-03-12.parquet', 'sentiment_features_load_date=2026-03-12.parquet', 'temporal_features_load_date=2026-03-12.parquet'])

In [7]:
check = dfs['ml_features_load_date=2026-03-12.parquet']

## Organizing Columns

In [8]:
check.columns

Index(['_index', 'product_id', 'product_parent', 'label', 'marketplace_id',
       'product_category_id', 'vine', 'verified_purchase', 'review_date',
       'detected_language', 'review_age_days', 'review_relative_rank',
       'product_review_count', 'days_since_first_review', 'is_early_review',
       'reviews_same_day', 'review_month', 'review_dayofweek',
       'reviews_per_day', 'body_word_count', 'headline_word_count',
       'type_token_ratio', 'avg_word_length', 'exclamation_count',
       'question_count', 'sentence_count_approx', 'html_entity_density',
       'paragraph_break_count', 'has_structured_body', 'body_lang_zscore',
       'body_cat_zscore', 'headline_body_ratio', 'exclamation_density',
       'question_density', 'vine_x_marketplace', 'verified_x_category',
       'headline_body_cosine_sim', 'title_body_cosine_sim',
       'body_embedding_norm', 'body_sentiment_label', 'body_sentiment_score',
       'body_sentiment_polarity', 'headline_sentiment_label',
       'head

In [9]:
print(check.columns.tolist())

['_index', 'product_id', 'product_parent', 'label', 'marketplace_id', 'product_category_id', 'vine', 'verified_purchase', 'review_date', 'detected_language', 'review_age_days', 'review_relative_rank', 'product_review_count', 'days_since_first_review', 'is_early_review', 'reviews_same_day', 'review_month', 'review_dayofweek', 'reviews_per_day', 'body_word_count', 'headline_word_count', 'type_token_ratio', 'avg_word_length', 'exclamation_count', 'question_count', 'sentence_count_approx', 'html_entity_density', 'paragraph_break_count', 'has_structured_body', 'body_lang_zscore', 'body_cat_zscore', 'headline_body_ratio', 'exclamation_density', 'question_density', 'vine_x_marketplace', 'verified_x_category', 'headline_body_cosine_sim', 'title_body_cosine_sim', 'body_embedding_norm', 'body_sentiment_label', 'body_sentiment_score', 'body_sentiment_polarity', 'headline_sentiment_label', 'headline_sentiment_score', 'headline_sentiment_polarity', 'sentiment_mismatch', 'repetition_ratio', 'flesch_

In [10]:
# Grouping column by type, two of which are in described in Notion
temporal_features = ['review_age_days', 'review_relative_rank', 'product_review_count', 'days_since_first_review', 'is_early_review', 'reviews_same_day', 'review_month', 'review_dayofweek']
language_features = ['detected_language', 'body_word_count', 'headline_word_count', 'body_lang_zscore', 'headline_body_ratio', 'type_token_ratio', 'avg_word_length', 'exclamation_density', 'question_density', 'vine_x_marketplace', 'verified_x_category']
all_features = temporal_features + language_features

leftover = [col for col in check.columns if col not in all_features]

print("Temporal features present:", all(f in check.columns for f in temporal_features))
print("Language features present:", all(f in check.columns for f in language_features))
print("Leftover columns:", leftover)

Temporal features present: True
Language features present: True
Leftover columns: ['_index', 'product_id', 'product_parent', 'label', 'marketplace_id', 'product_category_id', 'vine', 'verified_purchase', 'review_date', 'reviews_per_day', 'exclamation_count', 'question_count', 'sentence_count_approx', 'html_entity_density', 'paragraph_break_count', 'has_structured_body', 'body_cat_zscore', 'headline_body_cosine_sim', 'title_body_cosine_sim', 'body_embedding_norm', 'body_sentiment_label', 'body_sentiment_score', 'body_sentiment_polarity', 'headline_sentiment_label', 'headline_sentiment_score', 'headline_sentiment_polarity', 'sentiment_mismatch', 'repetition_ratio', 'flesch_reading_ease', 'headline_body_jaccard', 'title_body_jaccard', 'title_body_overlap', 'body_bigram_diversity', 'uppercase_ratio', 'digit_density', 'sentence_count', 'avg_sentence_length', 'flesch_lang_zscore', 'flesch_cat_zscore', 'polarity_lang_zscore', 'measurement_count', 'price_ref_count', 'ordinal_comparison_count',

In [11]:
check_subset = check[all_features + ['label']]

In [13]:
# Overview of values per column in check_subset
check_subset.describe(include='all')

,review_age_days,review_relative_rank,product_review_count,days_since_first_review,is_early_review,reviews_same_day,review_month,review_dayofweek,detected_language,body_word_count,headline_word_count,body_lang_zscore,headline_body_ratio,type_token_ratio,avg_word_length,exclamation_density,question_density,vine_x_marketplace,verified_x_category,label
count,11729.000000,11999.000000,11999.000000,11729.000000,11999,11999.000000,11729.000000,11729.000000,11999,11999.000000,11999.000000,1.199500e+04,11999.000000,11999.000000,11999.000000,11999.000000,11999.000000,11999,11999,9613
unique,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,59,2
top,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,en,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N_0.0,Y_3.0,False
freq,NaN,NaN,NaN,NaN,6264,NaN,NaN,NaN,3991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4824,2457,5139
mean,1165.072726,3.241020,5.482040,462.889590,NaN,1.014501,6.347941,2.925569,NaN,84.618218,3.545379,-9.477852e-18,0.147176,0.852095,4.919756,0.374603,0.028715,NaN,NaN,NaN
std,1243.582478,4.912749,8.071029,947.415824,NaN,0.131501,3.564608,1.995134,NaN,146.456988,3.031961,9.990407e-01,0.498978,0.118741,1.670430,2.122751,0.170696,NaN,NaN,NaN
min,0.000000,1.000000,1.000000,0.000000,NaN,1.000000,1.000000,0.000000,NaN,1.000000,0.000000,-1.884242e+00,0.000000,0.115385,1.000000,0.000000,0.000000,NaN,NaN,NaN
25%,366.000000,1.000000,1.000000,0.000000,NaN,1.000000,3.000000,1.000000,NaN,23.000000,2.000000,-4.082295e-01,0.029412,0.783784,4.400000,0.000000,0.000000,NaN,NaN,NaN
50%,751.000000,2.000000,3.000000,17.000000,NaN,1.000000,6.000000,3.000000,NaN,38.000000,3.000000,-3.116276e-01,0.058824,0.870968,4.850000,0.000000,0.000000,NaN,NaN,NaN
75%,1402.000000,3.000000,6.000000,438.000000,NaN,1.000000,9.000000,5.000000,NaN,87.000000,5.000000,2.325909e-02,0.131579,0.950000,5.333333,0.200000,0.000000,NaN,NaN,NaN


In [ ]:
# Missing values per column
check_subset.isnull().sum()

review_age_days             270
review_relative_rank          0
product_review_count          0
days_since_first_review     270
is_early_review               0
reviews_same_day              0
review_month                270
review_dayofweek            270
detected_language             0
body_word_count               0
headline_word_count           0
body_lang_zscore              4
headline_body_ratio           0
type_token_ratio              0
avg_word_length               0
exclamation_density           0
question_density              0
vine_x_marketplace            0
verified_x_category           0
label                      2386
dtype: int64

## ML Models

In [ ]:
# We are missing column dataset_split, so can't make any predictions yet
# Placeholder for Machine Learning model training and evaluation

In [ ]:
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split
# from xgboost import XGBClassifier

# # Assuming 'check' is the dataframe, 'all_features' are the feature columns, 'label' is the target
# X = check[all_features]
# y = check['label']

# # Split the data
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67)

# # Pipeline for Logistic Regression
# log_reg_pipeline = Pipeline([
#     ('scaler', StandardScaler()),
#     ('classifier', LogisticRegression(random_state=67, max_iter=1000))
# ])

# # Pipeline for XGBoost
# xgb_pipeline = Pipeline([
#     ('classifier', XGBClassifier(random_state=67, eval_metric='logloss'))
# ])

# # Training
# log_reg_pipeline.fit(X_train, y_train)
# xgb_pipeline.fit(X_train, y_train)

# # Predicting
# log_reg_pred = log_reg_pipeline.predict(X_test)
# xgb_pred = xgb_pipeline.predict(X_test)